# P-Task4 (Part B): HR Employee Ranker

**Problem:** HR chahta hai employees performance score ke hisaab se rank hon. Score barabar ho to experience
(zyada experience pehle), phir bhi barabar ho to naam alphabetically. Sorted list mein se kisi employee ko
naam se jaldi search bhi karna hai.

**Concepts used:** Big O (conceptual), Bubble / Selection / Insertion sort from scratch, `sort()` vs `sorted()`,
sorting list of dicts, binary search.

## Big O (aasan zabaan mein)
| Notation | Matlab | Misal |
|---|---|---|
| O(1) | data kitna bhi ho, ek step | list ka index se element lena |
| O(log n) | har step mein aadha data khatam | binary search |
| O(n) | data jitna, utne steps | linear search |
| O(n²) | har item ko har item se compare | bubble / selection / insertion sort |

## 1. Employees ka data

In [1]:
employees = [
    {"name": "Sana Iqbal",   "score": 88, "experience": 5},
    {"name": "Ali Raza",     "score": 92, "experience": 3},
    {"name": "Hina Tariq",   "score": 88, "experience": 7},
    {"name": "Bilal Ahmed",  "score": 75, "experience": 2},
    {"name": "Maryam Aslam", "score": 92, "experience": 3},
    {"name": "Usman Ghani",  "score": 81, "experience": 10},
    {"name": "Zara Sheikh",  "score": 75, "experience": 2},
    {"name": "Danish Khan",  "score": 88, "experience": 5},
]
len(employees)

8

## 2. Sorting ka tareeqa
Rank ke liye ek "key" banayi: `(-score, -experience, name)`. Score aur experience ke aage minus isliye ke
zyada wala pehle aaye, naam normal A-Z rehta hai.

In [2]:
def rank_key(emp):
    return (-emp["score"], -emp["experience"], emp["name"])

ranked = sorted(employees, key=rank_key)

print(f"{'Rank':<6}{'Name':<15}{'Score':>6}{'Exp':>5}")
for pos, e in enumerate(ranked, start=1):
    print(f"{pos:<6}{e['name']:<15}{e['score']:>6}{e['experience']:>5}")

Rank  Name            Score  Exp
1     Ali Raza           92    3
2     Maryam Aslam       92    3
3     Hina Tariq         88    7
4     Danish Khan        88    5
5     Sana Iqbal         88    5
6     Usman Ghani        81   10
7     Bilal Ahmed        75    2
8     Zara Sheikh        75    2


## 3. Sorting algorithms from scratch
Teen basic algorithms khud likhe. Teeno `key` function leti hain taake list of dicts bhi sort ho sake.

In [3]:
def bubble_sort(items, key):
    data = items[:]                      # copy, original na badle
    n = len(data)
    for i in range(n):
        swapped = False
        for j in range(n - i - 1):
            if key(data[j]) > key(data[j + 1]):
                data[j], data[j + 1] = data[j + 1], data[j]
                swapped = True
        if not swapped:                  # koi swap nahi hua to list sorted hai
            break
    return data


def selection_sort(items, key):
    data = items[:]
    for i in range(len(data)):
        smallest = i
        for j in range(i + 1, len(data)):
            if key(data[j]) < key(data[smallest]):
                smallest = j
        data[i], data[smallest] = data[smallest], data[i]
    return data


def insertion_sort(items, key):
    data = items[:]
    for i in range(1, len(data)):
        current = data[i]
        j = i - 1
        while j >= 0 and key(data[j]) > key(current):
            data[j + 1] = data[j]
            j -= 1
        data[j + 1] = current
    return data


expected = [e["name"] for e in ranked]
for algo in (bubble_sort, selection_sort, insertion_sort):
    result = [e["name"] for e in algo(employees, rank_key)]
    print(f"{algo.__name__:<15} same as sorted()? {result == expected}")

bubble_sort     same as sorted()? True
selection_sort  same as sorted()? True
insertion_sort  same as sorted()? True


## 4. `sort()` vs `sorted()`
* `sorted(list)` nayi list banata hai, original wesi hi rehti hai
* `list.sort()` usi list ko badal deta hai aur `None` return karta hai

In [4]:
numbers = [5, 2, 9, 1]
new_list = sorted(numbers)
print("sorted() ke baad:", numbers, "->", new_list)

result = numbers.sort()
print("sort() ke baad  :", numbers, "-> return value:", result)

sorted() ke baad: [5, 2, 9, 1] -> [1, 2, 5, 9]
sort() ke baad  : [1, 2, 5, 9] -> return value: None


## 5. Binary search (naam se)
Search ke liye list naam ke hisaab se sorted honi chahiye. Har step mein beech wala naam dekhte hain,
phir aadhi list chhod dete hain. Isliye O(log n).

In [5]:
by_name = sorted(employees, key=lambda e: e["name"].lower())


def binary_search(sorted_list, target):
    low, high = 0, len(sorted_list) - 1
    steps = 0
    while low <= high:
        steps += 1
        mid = (low + high) // 2
        current = sorted_list[mid]["name"].lower()
        if current == target:
            return mid, steps
        elif current < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1, steps


for name in ["Hina Tariq", "Zara Sheikh", "Ahmed Ali"]:
    index, steps = binary_search(by_name, name.lower())
    if index == -1:
        print(f"{name:<12} -> nahi mila ({steps} steps)")
    else:
        emp = by_name[index]
        rank = ranked.index(emp) + 1
        print(f"{name:<12} -> mil gaya ({steps} steps)  score={emp['score']}  rank={rank}")

Hina Tariq   -> mil gaya (1 steps)  score=88  rank=3
Zara Sheikh  -> mil gaya (4 steps)  score=75  rank=8
Ahmed Ali    -> nahi mila (3 steps)


## Conclusion
Teen keys (score, experience, name) ek tuple mein daal kar ranking ek hi `sorted()` call se ho gayi.
Teeno from-scratch algorithms ka result `sorted()` jaisa hi aaya. Binary search ne 8 employees mein se
naam sirf 1 se 4 steps mein dhoondh liya (ya bata diya ke nahi hai), jab ke linear search mein 8 tak lag sakte hain.